# Attention Key-Value Dynamics

How key and value representations evolve across layers: norm evolution,
K-V alignment, key similarity, and cross-layer summary.

## Why This Matters

Attention key value dynamics reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_key_value_dynamics import (
    key_norm_evolution, value_norm_evolution,
    key_value_alignment, key_similarity_across_positions,
    kv_dynamics_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Key Norm Evolution

How do key magnitudes change through the model?

In [ ]:
result = key_norm_evolution(model, tokens)
print(f"Trend: {result['norm_trend']}, max at layer {result['max_norm_layer']}\n")
for p in result['per_layer']:
    bar = '█' * int(p['mean_key_norm'] * 10)
    print(f"  Layer {p['layer']}: norm={p['mean_key_norm']:.4f} {bar}")

## Value Norm Evolution

Value norms indicate how much information each layer writes.

In [ ]:
result = value_norm_evolution(model, tokens)
print(f"Trend: {result['norm_trend']}, max at layer {result['max_norm_layer']}\n")
for p in result['per_layer']:
    bar = '█' * int(p['mean_value_norm'] * 10)
    print(f"  Layer {p['layer']}: norm={p['mean_value_norm']:.4f} {bar}")

## Key-Value Alignment

Do keys and values point in similar directions?

In [ ]:
for layer in range(4):
    result = key_value_alignment(model, tokens, layer=layer)
    flag = ' [ALIGNED]' if result['is_aligned'] else ''
    print(f"  Layer {layer}: mean_align={result['mean_alignment']:.4f}{flag}")
    for h in result['per_head']:
        print(f"    H{h['head']}: {h['mean_alignment']:.4f}")

## Key Similarity Across Positions

Are keys diverse (enabling selective attention) or uniform?

In [ ]:
for layer in range(4):
    for head in range(4):
        r = key_similarity_across_positions(model, tokens, layer, head)
        flag = ' [DIVERSE]' if r['is_diverse'] else ''
        print(f"  L{layer}H{head}: sim={r['mean_key_similarity']:.4f}{flag}")

## KV Dynamics Summary

Cross-layer overview of key-value dynamics.

In [ ]:
result = kv_dynamics_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: K={p['mean_key_norm']:.4f}, V={p['mean_value_norm']:.4f}, "
          f"align={p['mean_kv_alignment']:.4f}, ratio={p['kv_norm_ratio']:.4f}")